In [1]:
import os
os.environ['http_proxy'] = 'http://127.0.0.1:7896'
os.environ['https_proxy'] = 'http://127.0.0.1:7896'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7897'

from typing import Iterable, List, Optional, Tuple
import open_clip
import argparse
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
from transformers import CLIPVisionModel
from data_preparing.eegdatasets_joint_subjects import EEGDataset

from model.custom_pipeline import *
from model.FusedEEGViT import FusedEEGViT

/home/wenxiao/anaconda3/envs/BCI/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_trainable_parameters(model: FusedEEGViT) -> FusedEEGViT:
    """只训练 bridge + gate + brain adapter，其余保持冻结。"""

    # visual 在 __init__ 里已经冻结

    # 启用 bridges、gate_head、final_norm 的参数
    for m in [model.bridges, model.gate_head, model.final_norm]:
        for p in m.parameters():
            p.requires_grad = True

    # Brain 侧：根据需要选择性冻结/解冻
    for name, p in model.brain_adapter.named_parameters():
        p.requires_grad = True

    return model

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
bridge_pairs = [(6, 0), (12, 1), (18, 2)]
model = FusedEEGViT(
    bridge_pairs=bridge_pairs,
    n_heads_bridge=8,
    lora_r=8,
    med_depth=3,
    model_type="ViT-H-14",
    pretrained="laion2b_s32b_b79k",
).to(device)
set_trainable_parameters(model)
B = 1
device = "cuda:0" if torch.cuda.is_available() else "cpu"
_, _, preprocess = open_clip.create_model_and_transforms(
    model_name="ViT-H-14",
    pretrained="laion2b_s32b_b79k",
)
image_path = "/home/wenxiao/workspace/qhy/BMCA/image.png"
image = Image.open(image_path).convert("RGB")
pixel_values = preprocess(image).unsqueeze(0).to(device)
# pixel_values = torch.randn(B, 3, 224, 224, device=device)
eeg = torch.randn(B, 63, 250, device=device)
out = model(pixel_values, eeg)
print(out)
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape))
    else:
        print(k, v)

x: torch.Size([1, 257, 1280])
brain_tokens after patch_embed: torch.Size([1, 256, 1280])
x: torch.Size([1, 257, 1280])
vit_tokens torch.Size([1, 257, 1280])
tokens: torch.Size([1, 257, 1280])
[Medformer layer 0] input x: torch.Size([1, 256, 1280])
[Medformer layer 0] output out0: torch.Size([1, 256, 1280])
[Medformer layer 1] input x: torch.Size([1, 256, 1280])
[Medformer layer 1] output out0: torch.Size([1, 256, 1280])
[Medformer layer 2] input x: torch.Size([1, 256, 1280])
[Medformer layer 2] output out0: torch.Size([1, 256, 1280])
{'z_pure': tensor([[-0.0984, -0.5647,  0.1665,  ...,  0.0171, -0.1896,  0.8429]],
       device='cuda:0'), 'z_fused': tensor([[ 0.4624,  0.2951, -0.6237,  ..., -0.0457,  0.1205, -0.5809]],
       device='cuda:0', grad_fn=<MmBackward0>), 'z_gate': tensor([[-0.0314, -0.2183,  0.1620,  ...,  0.0396, -0.1242,  0.4759]],
       device='cuda:0', grad_fn=<AddBackward0>), 'gate': tensor([0.5525], device='cuda:0', grad_fn=<SqueezeBackward1>), 'brain_tokens': tensor

In [4]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

generator = Generator4Embeds()
z_pure = out["z_pure"]  # Tensor
z_pure = z_pure / z_pure.norm(dim=-1, keepdim=True)  # 归一化
z_pure = z_pure.unsqueeze(0)

# 需要确保 device/dtype 与 generator 的一致
z_pure = z_pure.to(device=device, dtype=generator.dtype)

# 如果 batch_size=1，则取第一个即可
if z_pure.dim() == 4:   # (B, C, H, W)
    z_pure = z_pure[0]

# 生成图像
image = generator.generate(
    image_embeds=z_pure, 
    text_prompt="", 
    generator=None
)

# 保存或显示图像
image.save("generated.png")

  0%|          | 0/1 [00:00<?, ?it/s]/home/wenxiao/anaconda3/envs/BCI/lib/python3.10/site-packages/diffusers/models/embeddings.py:2587: FutureWarning: You have passed a tensor as `image_embeds`.This is deprecated and will be removed in a future release. Please make sure to update your script to pass `image_embeds` as a list of tensors to suppress this warning.
  deprecate("image_embeds not a list", "1.0.0", deprecation_message, standard_warn=False)
100%|██████████| 1/1 [00:00<00:00,  2.06it/s]
